## Robust design

An outer search over the nav2 settings we choose, and for each one an inner adversary over
the doorway and the walker we do not. A tuning's score is the worst its adversary could
find, so the winner is the one whose bad day is least bad — not the one that looks best on
average.

**What to look for:** `report().extra['robust_tuning']` in the campaign record holds the
answer. The controller's own "best objective" does **not**: it folds the flat inner
objective, which is a different quantity.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json
import pandas as pd
import matplotlib.pyplot as plt

from robovast.common.analysis import CampaignDataError, open_campaign_store

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from the campaign's own store rather than the results index because that is where a
    SEARCH records what it scored -- the index holds per-run tables, and a search's unit of
    analysis is the cell. The store is also written as the search runs, so this works on a
    campaign that is still going or was never postprocessed.
    """
    # open_campaign_store rather than a sqlite3.connect on a path built here: it resolves the
    # campaign ROOT from data_dir, so this cell also works at a configuration node instead of
    # only at the campaign, and it is the one place that knows where the store lives.
    try:
        conn = open_campaign_store(data_dir)
    except CampaignDataError as exc:
        # Reported, not swallowed. "This campaign scored nothing" and "its record is not
        # here" are different answers and only the first is a result -- an empty frame
        # returned quietly reads as the first while meaning the second.
        print(f'[no data] {exc}')
        return pd.DataFrame()
    try:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    finally:
        conn.close()
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Minimax — which tuning survives the worst'

# Clearance against time. The highlighted points are the trade-offs actually available:
# nothing else beats them on both at once.
if not scored.empty and 'm_time_to_goal' in scored:
    x, y = scored['m_min_clearance'], scored['m_time_to_goal']
    on_front = [not ((x > xi) & (y < yi)).any() for xi, yi in zip(x, y)]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(x, y, s=60, alpha=0.4, label='evaluated')
    ax.scatter(x[on_front], y[on_front], s=120, edgecolor='black', linewidth=0.8,
               color='tab:orange', label='non-dominated')
    ax.set_xlabel('minimum clearance [m]  (more is safer)')
    ax.set_ylabel('time to goal [s]  (less is better)')
    ax.set_title('%s: what safety costs' % TITLE)
    ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print()
    outer = [c for c in ('inflation_radius', 'max_speed') if c in scored]
    print("What this campaign is FOR -- which TUNING survives the worst:")
    if outer:
        worst_per_tuning = scored.groupby(outer)['robustness'].min()
        best_tuning = worst_per_tuning.idxmax()
        print(f"  tunings evaluated    : {len(worst_per_tuning)}")
        print(f"  most robust tuning   : " +
              ", ".join(f"{c}={v:.3f}"
                        for c, v in zip(outer, best_tuning if isinstance(best_tuning, tuple)
                                        else (best_tuning,))))
        print(f"  its worst crossing   : {worst_per_tuning.max():.3f}")
        print(f"  the worst tuning's   : {worst_per_tuning.min():.3f}")
        print("  The gap between those two is what the tuning choice is worth.")
